In [1]:
from omegaconf import OmegaConf
import torch
from PIL import Image
import numpy as np

from utils.qwen_image_edit_wrapper import QwenImageEditWrapper, calculate_dimensions
from diffusers import QwenImageEditPlusPipeline
from utils.scheduler import FlowMatchScheduler, SchedulerInterface

from diffusers.utils import make_image_grid
import inspect
from pathlib import Path


from utils.dataset import ImageEditDataset, cycle

/home/rick-mbp/miniconda3/envs/unified/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
CONFIG_PATH = "/home/rick-mbp/Diffusion-Distillation/configs/qwen_dmd.yaml"

DATASET_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_image_edit_dataset_v3.0.1/metadata_edit.csv"
)

In [3]:
config = OmegaConf.load(CONFIG_PATH)

model_name = config["real_name"]

teacher_lora_path = config["teacher_lora_path"]

In [4]:
dataset = ImageEditDataset(DATASET_PATH)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, num_workers=8)
dataloader = cycle(dataloader)

In [5]:
qwen_wrapper = QwenImageEditWrapper(
    model_name=model_name,
    teacher_lora_path=teacher_lora_path,  # Put path here if you have it
).to("cuda")
# qwen_wrapper.eval()

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  4.00it/s]


Loading Teacher LoRA weights from: /mnt/model_training_disk/qwen_model_weight/ckpt/Qwen-Image-Edit-2509_lora-rank-32_lr-1e-4_hard/step-6000-hf.safetensors


/home/rick-mbp/miniconda3/envs/unified/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# Test LoRA

qwen_wrapper.set_adapter_trainable("generator")

base_trainable = 0
generator_trainable = 0
fake_trainable = 0
real_trainable = 0

model = qwen_wrapper.pipe.transformer


for name, param in model.named_parameters():
    if param.requires_grad:
        if "generator" in name:
            generator_trainable += 1
            # Print first instance to verify
            if generator_trainable == 1:
                print(f"✅ Generator is TRAINABLE ({name})")
        elif "fake" in name:
            fake_trainable += 1
            if fake_trainable == 1:
                print(f"✅ Fake is TRAINABLE ({name})")
        elif "real" in name:
            real_trainable += 1
            print(f"❌ WARNING: Real is TRAINABLE ({name})")
        else:
            base_trainable += 1
            # Print first instance to verify
            if base_trainable == 1:
                print(f"⚠️ Base Model is TRAINABLE ({name})")

print("-" * 60)
print(f"Summary of params with requires_grad=True:")
print(f"Base Model params: {base_trainable} (Should be 0)")
print(f"Real LoRA params:  {real_trainable} (Should be 0)")
print(f"Fake LoRA params:  {fake_trainable} (Should be >0 if training fake)")
print(f"Gen LoRA params:   {generator_trainable} (Should be >0 if training gen)")
print(f"{'=' * 60}\n")

### Test with dataloader

In [6]:
import torch
import math
from PIL import Image
from tqdm import tqdm  # For the progress bar
import torchvision.transforms.functional as TF

# Assuming wrapper is initialized
# wrapper = QwenImageEditWrapper(...)
qwen_wrapper.switch_to_real()


batch = next(dataloader)

img = batch["img"]
prompt = batch["prompts"]

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

num_inference_steps = 20
guidance_scale = (
    1.0  # Common for distilled models, controls how strongly to follow text
)
device = "cuda"
dtype = torch.bfloat16


VAE_IMAGE_SIZE = 1024 * 1024
CONDITION_IMAGE_SIZE = 384 * 384


condition_image_sizes = []
condition_images = []
vae_image_sizes = []
vae_images = []


image_width, image_height = img.shape[-2:]


vae_width, vae_height = calculate_dimensions(VAE_IMAGE_SIZE, image_width / image_height)
condition_width, condition_height = calculate_dimensions(
    CONDITION_IMAGE_SIZE, image_width / image_height
)


condition_image_sizes.append((condition_width, condition_height))
vae_image_sizes.append((vae_width, vae_height))


condition_images.append(
    qwen_wrapper.pipe.image_processor.resize(img, condition_height, condition_width)
)
vae_images.append(
    qwen_wrapper.pipe.image_processor.preprocess(img, vae_height, vae_width)
    .to(device=device, dtype=dtype)
    .unsqueeze(2)
)

# ---------------------------------------------------------
# 2. Prepare Condition Latents (The Reference Image)
# ---------------------------------------------------------
print(f">>> Encoding Reference Image...")
with torch.no_grad():
    # Encode
    all_image_latents = []
    for vae_img in vae_images:
        image_latent = qwen_wrapper.encode_vae_img(vae_img)
        all_image_latents.append(image_latent)

    image_latents = torch.cat(all_image_latents, dim=1)
    image_latent_height, image_latent_width = image_latents.shape[3:]


# Convert 2D Grid [B, C, H, W] -> Sequence [B, Seq, C] for Transformer
clean_latents_seq = qwen_wrapper.pipe._pack_latents(
    image_latents,
    batch_size=1,
    num_channels_latents=qwen_wrapper.pipe.latent_channels,
    width=image_latent_width,
    height=image_latent_height,
)


# This is our conditioning reference (concatenated in wrapper)
image_latents_condition = clean_latents_seq

# ---------------------------------------------------------
# 3. Prepare Text Embeddings
# ---------------------------------------------------------
print(f">>> Encoding Prompt: '{prompt}'...")
with torch.no_grad():
    prompt_embeds, prompt_masks = qwen_wrapper.pipe.encode_prompt(
        prompt=prompt,
        device=device,
        image=condition_images,
    )

cond_dict = {
    "prompt_embeds": prompt_embeds.to(dtype),
    "prompt_embeds_mask": prompt_masks.to(device),
}

# ---------------------------------------------------------
# 4. Initialize Scheduler and Noise
# ---------------------------------------------------------
print(f">>> Setting up Scheduler ({num_inference_steps} steps)...")
# Setup timesteps (e.g., [1000, 950, 900 ... 0])
qwen_wrapper.scheduler.set_timesteps(num_inference_steps)
timesteps = qwen_wrapper.scheduler.timesteps

# Start with Pure Random Noise (x_T)
# Shape must match the packed sequence shape
latents = torch.randn_like(clean_latents_seq)

# ---------------------------------------------------------
# 5. Denoising Loop
# ---------------------------------------------------------
print(f">>> Starting Denoising Loop...")

with torch.no_grad():
    for i, t in tqdm(enumerate(timesteps), total=len(timesteps)):
        # Expand timestep to batch size [1]
        timestep_tensor = t.expand(latents.shape[0]).to(device)

        # A. Predict Flow/Noise (Wrapper Forward)
        # Note: We pass 'clean_latents_seq' as 'image_latents'.
        # This tells Qwen: "Edit THIS image (clean) to match the prompt."
        flow_pred, _ = qwen_wrapper.forward(
            noisy_latents=latents,
            conditional_dict=cond_dict,
            timestep=timestep_tensor,
            guidance_scale=guidance_scale,
            image_latents=image_latents_condition,
            height=vae_height,
            width=vae_width,
        )

        # B. Step (Scheduler)
        # Computes x_{t-1} = x_t + dt * v_t
        step_output = qwen_wrapper.scheduler.step(
            flow_pred, t.unsqueeze(0).to(device), latents
        )
        latents = step_output[0].to(torch.bfloat16)

        break


# ---------------------------------------------------------
# 6. Decode Final Result
# ---------------------------------------------------------
print(f">>> Decoding Final Image...")
with torch.no_grad():
    # A. Unpack Sequence -> 2D
    latents_2d = qwen_wrapper._unpack_latents(
        latents,  # These are the final denoised latents
        height=vae_height,
        width=vae_width,
    )

    image_tensor = qwen_wrapper.decode_vae_latent(latents_2d)

    # D. Postprocess
    final_image = qwen_wrapper.pipe.image_processor.postprocess(
        image_tensor, output_type="pil"
    )[0]


Switching to Real Score
>>> Encoding Reference Image...
>>> Encoding Prompt: '['clean room']'...
>>> Setting up Scheduler (20 steps)...
>>> Starting Denoising Loop...


  0%|          | 0/20 [00:00<?, ?it/s]

noisy_latents: torch.Size([1, 4096, 64])
image_latents: torch.Size([1, 4096, 64])
latent_height: 128
latent_width: 128
latent_model_input: torch.Size([1, 8192, 64])
transformer_timestep: torch.Size([1])
guidance: None
prompt_embeds: torch.Size([1, 210, 3584])
prompt_embeds_mask: torch.Size([1, 210])
txt_seq_lens: [210]


  0%|          | 0/20 [00:00<?, ?it/s]

>>> Decoding Final Image...


In [ ]:
 latents.shape, latents_2d.shape, image_tensor.shape


(torch.Size([1, 4096, 64]),
 torch.Size([1, 4096, 64]),
 torch.Size([1, 16, 1, 128, 128]),
 torch.Size([1, 3, 1024, 1024]))

In [ ]:
img_tensor = img.detach().cpu().squeeze(0)
img_np = img_tensor.permute(1, 2, 0).numpy()

img_np = (img_np * 255).astype(np.uint8)


original_image = Image.fromarray(img_np)

make_image_grid([final_image, original_image], cols=2, rows=1)


### Test compute score

In [ ]:
noisy_img = torch.randn_like(latents_2d)


qwen_wrapper.compute_score(noisy_img, cond_dict, timestep_tensor, latents_2d)

In [ ]:
cond_dict["prompt_embeds_mask"].shape

### new end to end test


In [ ]:
import torch
import math
from PIL import Image
from tqdm import tqdm  # For the progress bar

# Assuming wrapper is initialized
# wrapper = QwenImageEditWrapper(...)
qwen_wrapper.switch_to_real()

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
img_path = "/home/rick-mbp/Diffusion-Distillation/test_data/019ab527-755e-70ec-9ec4-978ced8b543a.jpg"
prompt = "empty room"  # The edit instruction
num_inference_steps = 20
guidance_scale = (
    1.0  # Common for distilled models, controls how strongly to follow text
)
device = "cuda"
dtype = torch.bfloat16


VAE_IMAGE_SIZE = 1024 * 1024
CONDITION_IMAGE_SIZE = 384 * 384


# ---------------------------------------------------------
# 1. Load and Preprocess Real Image
# ---------------------------------------------------------
print(f">>> Loading Image...")
original_image = Image.open(img_path).convert("RGB").resize((1024, 1024))

image_width, image_height = original_image.size
aspect_ratio = image_width / image_height
original_image = TF.to_tensor(original_image).unsqueeze(0)


condition_image_sizes = []
condition_images = []
vae_image_sizes = []
vae_images = []


vae_width, vae_height = calculate_dimensions(VAE_IMAGE_SIZE, image_width / image_height)
condition_width, condition_height = calculate_dimensions(
    CONDITION_IMAGE_SIZE, image_width / image_height
)


condition_image_sizes.append((condition_width, condition_height))
vae_image_sizes.append((vae_width, vae_height))


condition_images.append(
    qwen_wrapper.pipe.image_processor.resize(
        original_image, condition_height, condition_width
    )
)
vae_images.append(
    qwen_wrapper.pipe.image_processor.preprocess(original_image, vae_height, vae_width)
    .to(device=device, dtype=dtype)
    .unsqueeze(2)
)

# ---------------------------------------------------------
# 2. Prepare Condition Latents (The Reference Image)
# ---------------------------------------------------------
print(f">>> Encoding Reference Image...")
with torch.no_grad():
    # Encode
    all_image_latents = []
    for vae_img in vae_images:
        image_latent = qwen_wrapper.encode_vae_img(vae_img)
        all_image_latents.append(image_latent)

    image_latents = torch.cat(all_image_latents, dim=1)
    image_latent_height, image_latent_width = image_latents.shape[3:]


# Convert 2D Grid [B, C, H, W] -> Sequence [B, Seq, C] for Transformer
clean_latents_seq = qwen_wrapper.pipe._pack_latents(
    image_latents,
    batch_size=1,
    num_channels_latents=qwen_wrapper.pipe.latent_channels,
    width=image_latent_width,
    height=image_latent_height,
)


# This is our conditioning reference (concatenated in wrapper)
image_latents_condition = clean_latents_seq

# ---------------------------------------------------------
# 3. Prepare Text Embeddings
# ---------------------------------------------------------
print(f">>> Encoding Prompt: '{prompt}'...")
with torch.no_grad():
    prompt_embeds, prompt_masks = qwen_wrapper.pipe.encode_prompt(
        prompt=prompt,
        device=device,
        image=condition_images,
    )

cond_dict = {
    "prompt_embeds": prompt_embeds.to(dtype),
    "prompt_embeds_mask": prompt_masks.to(device),
}

# ---------------------------------------------------------
# 4. Initialize Scheduler and Noise
# ---------------------------------------------------------
print(f">>> Setting up Scheduler ({num_inference_steps} steps)...")
# Setup timesteps (e.g., [1000, 950, 900 ... 0])
qwen_wrapper.scheduler.set_timesteps(num_inference_steps)
timesteps = qwen_wrapper.scheduler.timesteps

# Start with Pure Random Noise (x_T)
# Shape must match the packed sequence shape
latents = torch.randn_like(clean_latents_seq)

# ---------------------------------------------------------
# 5. Denoising Loop
# ---------------------------------------------------------
print(f">>> Starting Denoising Loop...")

with torch.no_grad():
    for i, t in tqdm(enumerate(timesteps), total=len(timesteps)):
        # Expand timestep to batch size [1]
        timestep_tensor = t.expand(latents.shape[0]).to(device)

        # A. Predict Flow/Noise (Wrapper Forward)
        # Note: We pass 'clean_latents_seq' as 'image_latents'.
        # This tells Qwen: "Edit THIS image (clean) to match the prompt."
        flow_pred, _ = qwen_wrapper.forward(
            noisy_latents=latents,
            conditional_dict=cond_dict,
            timestep=timestep_tensor,
            guidance_scale=guidance_scale,
            image_latents=image_latents_condition,
            height=vae_height,
            width=vae_width,
        )

        # B. Step (Scheduler)
        # Computes x_{t-1} = x_t + dt * v_t
        step_output = qwen_wrapper.scheduler.step(
            flow_pred, t.unsqueeze(0).to(device), latents
        )
        latents = step_output[0].to(torch.bfloat16)

# ---------------------------------------------------------
# 6. Decode Final Result
# ---------------------------------------------------------
print(f">>> Decoding Final Image...")
with torch.no_grad():
    # A. Unpack Sequence -> 2D
    latents_2d = qwen_wrapper._unpack_latents(
        latents,  # These are the final denoised latents
        height=vae_height,
        width=vae_width,
        vae_scale_factor=qwen_wrapper.pipe.vae_scale_factor,
    )

    image_tensor = qwen_wrapper.decode_vae_latent(latents_2d)

    # D. Postprocess
    final_image = qwen_wrapper.pipe.image_processor.postprocess(
        image_tensor, output_type="pil"
    )[0]


In [ ]:
original_image = Image.open(img_path).convert("RGB").resize((1024, 1024))

make_image_grid([final_image, original_image], cols=2, rows=1)

### Test ImageEditTrainingPipeline

In [ ]:
from typing import List, Optional, Tuple

import torch
import torch.distributed as dist

from utils.qwen_image_edit_wrapper import QwenImageEditWrapper
from utils.scheduler import SchedulerInterface


class ImageEditTrainingPipeline:
    def __init__(
        self,
        model_name: str,
        denoising_step_list: List[int],
        scheduler: SchedulerInterface,
        generator: QwenImageEditWrapper,
    ):
        self.model_name = model_name
        self.scheduler = scheduler
        self.generator = generator
        self.denoising_step_list = denoising_step_list

        # Ensure step 0 is removed for inference trajectory to avoid numerical instability at t=0
        if self.denoising_step_list[-1] == 0:
            self.denoising_step_list = self.denoising_step_list[:-1]

    def _get_random_exit_step(self, num_denoising_steps, device):
        """
        Randomly select a timestep to stop the backward simulation.
        Syncs across ranks to ensure all GPUs stop at the same step for the batch.
        """
        rank = dist.get_rank() if dist.is_initialized() else 0
        if rank == 0:
            # Pick one random index for the whole batch
            exit_index = torch.randint(
                low=0, high=num_denoising_steps, size=(1,), device=device
            )
        else:
            exit_index = torch.empty(1, dtype=torch.long, device=device)

        if dist.is_initialized():
            dist.broadcast(exit_index, src=0)

        return exit_index.item()

    def infer(
        self,
        curr_noisy_input,
        cond_dict,
        timestep_tensor,
        guidance_scale,
        image_latents_condition,
    ):
        a, b = self.generator(
            noisy_latents=curr_noisy_input,
            conditional_dict=cond_dict,
            timestep=timestep_tensor,
            guidance_scale=guidance_scale,
            image_latents=image_latents_condition,
            height=1024,
            width=1024,
        )

    def inference_with_trajectory(
        self,
        noise: torch.Tensor,
        conditional_dict: dict,
        source_image_latent: Optional[torch.Tensor] = None,
        pixel_height: int = 1024,
        pixel_width: int = 1024,
    ) -> Tuple[torch.Tensor, int, int]:
        """
        Simulate the image editing trajectory for DMD/Consistency training.

        Input:
            - noise: [B, Seq_Len, C] - The noisy input.
            - source_image_latent: [B, Seq_Len, C] - The reference/condition image (clean).
                           Required for Image Edit tasks.
        """

        if noise.dim() == 3:
            batch_size, seq_len, num_channels = noise.shape

        else:
            raise ValueError(f"Unsupported noise shape: {noise.shape}")

        # 2. Determine where to stop the simulation (The "Self-Forcing" part)
        num_denoising_steps = len(self.denoising_step_list)
        exit_step_index = self._get_random_exit_step(
            num_denoising_steps, device=noise.device
        )

        curr_noisy_input = noise.clone()

        timestep_tensor = torch.full(
            (batch_size,),
            1,
            device=noise.device,
            dtype=torch.long,
        )

        # 3. Denoising Loop
        for index, current_timestep_val in enumerate(self.denoising_step_list):
            # Create timestep tensor [B]
            timestep_tensor = torch.full(
                (batch_size,),
                current_timestep_val,
                device=noise.device,
                dtype=torch.long,
            )

            # Check if this is our stop point
            is_exit_step = index == exit_step_index

            # --- Generator Call ---
            with torch.set_grad_enabled(is_exit_step):
                # We call the Qwen Wrapper.
                # Note: We pass 'pixel_height/width' not latent dimensions.

                print(f"curr_noisy_input: {curr_noisy_input.shape}")
                model_output = self.generator(
                    noisy_latents=curr_noisy_input,
                    conditional_dict=conditional_dict,
                    timestep=timestep_tensor,
                    height=pixel_height,
                    width=pixel_width,
                    image_latents=source_image_latent,
                )
                # Qwen Wrapper returns (flow_pred, pred_x0)
                # We need pred_x0 for consistency re-noising or for the final loss
                _, pred_x0 = model_output

                # Ensure pred_x0 is correct shape/type just in case
                denoised_pred = pred_x0

            if is_exit_step:
                # --- EXIT ---
                # We have reached the random step. Return the generator's x0 prediction.
                # The outer loop will compare this 'denoised_pred' against the real target image (DMD Loss).

                # Calculate timestep indices for logging
                # (Find where current_timestep_val sits in the full 0-1000 scheduler)
                denoised_timestep_from = (
                    1000
                    - torch.argmin(
                        (
                            self.scheduler.timesteps.to(noise.device)
                            - current_timestep_val
                        ).abs(),
                        dim=0,
                    ).item()
                )

                if index == len(self.denoising_step_list) - 1:
                    denoised_timestep_to = 0
                else:
                    next_t = self.denoising_step_list[index + 1]
                    denoised_timestep_to = (
                        1000
                        - torch.argmin(
                            (self.scheduler.timesteps.to(noise.device) - next_t).abs(),
                            dim=0,
                        ).item()
                    )

                return denoised_pred, denoised_timestep_from, denoised_timestep_to

            else:
                # --- CONTINUE ---
                # Trajectory Consistency: Use the predicted x0 to sample x_{t-1} (or x_{next_step})
                # This simulates the error accumulation of the model.
                with torch.no_grad():
                    next_timestep_val = self.denoising_step_list[index + 1]

                    # Prepare timesteps for scheduler [B]
                    next_timestep_tensor = torch.full(
                        (batch_size,),
                        next_timestep_val,
                        device=noise.device,
                        dtype=torch.long,
                    )

                    # Add noise back to the predicted x0 to get x_{next_step}
                    # FlowMatchScheduler.add_noise(original_samples, noise, timesteps)
                    # Result is x_t = (1-sigma)x0 + sigma*epsilon (for Rectified Flow)

                    curr_noisy_input = self.scheduler.add_noise(
                        denoised_pred,  # pred_x0
                        torch.randn_like(denoised_pred),  # new gaussian noise
                        next_timestep_tensor,
                    )
                    # TODO: current add noise is workeing on img not latent, we might can convert input of wrapper be image not latent

                    curr_noisy_input = curr_noisy_input.squeeze(0)
        # Fallback (should be covered by exit_step logic)
        return curr_noisy_input.unsqueeze(1), 0, 0


In [ ]:
inference_pipeline = ImageEditTrainingPipeline(
    model_name="generator",
    denoising_step_list=config["denoising_step_list"],
    scheduler=qwen_wrapper.get_scheduler(),
    generator=qwen_wrapper,
)


In [ ]:
denoised_pred_out, denoised_timestep_from, denoised_timestep_to = (
    inference_pipeline.inference_with_trajectory(
        noise=latents,
        source_image_latent=image_latents_condition,
        conditional_dict=cond_dict,
    )
)


In [ ]:
denoised_timestep_from, denoised_timestep_to


In [ ]:
denoised_pred_out.shape

In [ ]:
curr_noisy_input = latents.clone()
timestep_tensor = torch.full(
    (1,),
    1,
    device=latents.device,
    dtype=torch.long,
)


inference_pipeline.infer(
    curr_noisy_input, cond_dict, timestep_tensor, 10, image_latents_condition
)

In [ ]:
timestep_tensor = torch.full(
    (1,),
    1,
    device=latents.device,
    dtype=torch.long,
)


curr_noisy_input = latents.clone()

with torch.set_grad_enabled(True):
    a, b = qwen_wrapper(
        noisy_latents=curr_noisy_input,
        conditional_dict=cond_dict,
        timestep=timestep_tensor,
        guidance_scale=guidance_scale,
        image_latents=image_latents_condition,
        height=vae_height,
        width=vae_width,
    )

In [ ]:
b

In [ ]:
print(f"noisy_latents: {latents.shape}")
print(f"conditional_dict: {cond_dict}")
print(f"timestep: {timestep_tensor}")
print(f"height: {vae_height}")
print(f"width: {vae_width}")
print(f"image_latents: {image_latents_condition.shape}")

In [ ]:
timestep_tensor.dtype
